<a href="https://colab.research.google.com/github/naufallingzhi/ProjekStatistikaDeskriptifKel6/blob/main/Kelompok_6_Dashboard_Statistika_Deskriptif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dashboard Benchmark CS2 — Statistika Deskriptif


In [ ]:
from __future__ import annotations

"""Dashboard benchmark CS2 dengan fokus statistika deskriptif.

Desain mempertahankan tema Gaming Amber (hitam/abu-abu dengan aksen kuning)
serta komponen khas notebook sebelumnya: section bar, tabel gelap yang dapat
filter/sort, dropdown metrik, dan radio box jenis grafik.

Tidak ada pengujian statistika inferensial di dalam program ini.
"""

import html
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display

# ============================================================
# 1. KONFIGURASI UMUM DAN PALET WARNA ASLI
# ============================================================
URUTAN_GRAFIK = ["Low", "Medium", "High", "Very High"]
WARNA_GRAFIK = {
    "Low": "#FACC15",       # kuning neon
    "Medium": "#F59E0B",    # amber/gold
    "High": "#EA580C",      # oranye
    "Very High": "#EF4444", # merah
}
TEXT_GRAFIK = {
    "Low": "#111111",
    "Medium": "#111111",
    "High": "#FFFFFF",
    "Very High": "#FFFFFF",
}
KOLOM_NUMERIK = ["Min", "Max", "Avg", "Low1"]
LABEL_METRIK = {
    "Min": "Minimum FPS",
    "Max": "Maximum FPS",
    "Avg": "Average FPS",
    "Low1": "1% Low FPS",
}
NAMA_KOLOM_SUMBER = {
    "No": "No",
    "Jenis_Grafik": "Jenis_Grafik",
    "Min": "MinimumFramePerSeconds",
    "Max": "MaximumFramePerSeconds",
    "Avg": "AverageFramePerSeconds",
    "Low1": "1%FramesPerSecond",
}

# Urutan ukuran pada tabel mengikuti komponen statistika deskriptif dosen.
METRIK_STATISTIK = [
    "Jumlah (Sum)", "Count (N)", "Mean", "Modus", "Median",
    "Standard Error", "Standar Deviasi", "Varians",
    "Q1", "Q2", "Q3", "IQR", "Skewness", "Kurtosis (Excess)",
    "Minimum", "Maksimum", "Range", "CV (%)",
]

# ============================================================
# 2. PEMILIHAN FILE DAN PEMBACAAN DATASET
# ============================================================
def normalisasi_nama_kolom(nama: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(nama).strip().lower())


def pilih_file_dataset() -> str:
    """Choose Files di Colab; file chooser lokal bila dijalankan sebagai .py."""
    # Di Google Colab, gunakan upload yang dikenal pengguna.
    try:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        supported = (".xlsx", ".xls", ".xlsm", ".csv", ".tsv")
        filename = next(
            (name for name in uploaded if name.lower().endswith(supported)), None
        )
        if filename is None:
            raise ValueError(
                "File tidak didukung. Pilih .xlsx, .xls, .xlsm, .csv, atau .tsv."
            )
        return filename
    except ImportError:
        pass

    # Script lokal dapat menerima path sebagai argumen: python script.py data.xlsx
    for arg in sys.argv[1:]:
        if arg.lower().endswith((".xlsx", ".xls", ".xlsm", ".csv", ".tsv")) and os.path.isfile(arg):
            return arg

    # Coba dialog file desktop bila tkinter tersedia.
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        filename = filedialog.askopenfilename(
            title="Pilih dataset benchmark CS2",
            filetypes=[
                ("Dataset yang didukung", "*.xlsx *.xls *.xlsm *.csv *.tsv"),
                ("Excel", "*.xlsx *.xls *.xlsm"),
                ("CSV/TSV", "*.csv *.tsv"),
                ("Semua file", "*.*"),
            ],
        )
        root.destroy()
        if filename:
            return filename
    except Exception:
        pass

    filename = input("Masukkan lokasi dataset Excel/CSV/TSV: ").strip().strip('"')
    if not filename:
        raise ValueError("File dataset belum dipilih.")
    return filename


def baca_dataset(path: str, sheet_preferensi: str = "Dataset") -> tuple[pd.DataFrame, str | None, dict[str, str]]:
    """Baca Excel/CSV/TSV, normalisasi nama kolom, dan catat nama aslinya."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"File tidak ditemukan: {path}")

    ekstensi = Path(path).suffix.lower()
    sheet_dipakai: str | None = None

    if ekstensi in (".xlsx", ".xls", ".xlsm"):
        excel = pd.ExcelFile(path)
        sheets = excel.sheet_names
        if not sheets:
            raise ValueError("Workbook Excel tidak memiliki sheet.")
        if sheet_preferensi in sheets:
            sheet_dipakai = sheet_preferensi
        elif len(sheets) == 1:
            sheet_dipakai = sheets[0]
        else:
            print("Sheet tersedia:", ", ".join(sheets))
            pilihan = input(f"Masukkan nama sheet (Enter untuk {sheets[0]}): ").strip()
            sheet_dipakai = pilihan if pilihan in sheets else sheets[0]
        data = pd.read_excel(excel, sheet_name=sheet_dipakai)
    elif ekstensi == ".tsv":
        data = pd.read_csv(path, sep="\t")
    elif ekstensi == ".csv":
        data = pd.read_csv(path)
    else:
        raise ValueError("Format file tidak didukung.")

    kolom_asli = [str(c).strip() for c in data.columns]
    data.columns = kolom_asli

    alias = {
        "no": "No", "nomor": "No", "nomorurut": "No", "idpengujian": "No",
        "jenisgrafik": "Jenis_Grafik", "presetgrafik": "Jenis_Grafik", "preset": "Jenis_Grafik",
        "minimumframeperseconds": "Min", "minimumframespersecond": "Min", "minimumfps": "Min", "min": "Min",
        "maximumframeperseconds": "Max", "maximumframespersecond": "Max", "maximumfps": "Max", "max": "Max",
        "averageframeperseconds": "Avg", "averageframespersecond": "Avg", "averagefps": "Avg", "avg": "Avg",
        "1framespersecond": "Low1", "1framesperseconds": "Low1",
        "1lowfps": "Low1", "low1": "Low1", "1percentlowfps": "Low1",
    }
    rename_map: dict[str, str] = {}
    source_name_map: dict[str, str] = {}
    for col in data.columns:
        key = normalisasi_nama_kolom(col)
        if key in alias:
            rename_map[col] = alias[key]
            source_name_map[alias[key]] = str(col)
    data = data.rename(columns=rename_map)

    required = ["No", "Jenis_Grafik", *KOLOM_NUMERIK]
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise ValueError(
            f"Kolom dataset tidak ditemukan setelah pemeriksaan header: {missing}. "
            "Pastikan kolom No, Jenis_Grafik, Minimum, Maximum, Average, dan 1% Low FPS tersedia."
        )

    data = data[required].copy()
    return data, sheet_dipakai, source_name_map

# ============================================================
# 3. VALIDASI DAN PERSIAPAN DATA
# ============================================================
def validasi_dataset(data: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = data.copy()
    peringatan: list[str] = []
    df["Jenis_Grafik"] = df["Jenis_Grafik"].astype(str).str.strip()

    # Samakan kapitalisasi agar "low" dan "Low" dikenali sebagai preset yang sama.
    label_preset = {name.lower(): name for name in URUTAN_GRAFIK}
    df["Jenis_Grafik"] = df["Jenis_Grafik"].map(
        lambda v: label_preset.get(v.lower(), v)
    )
    for col in ["No", *KOLOM_NUMERIK]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    required = ["No", "Jenis_Grafik", *KOLOM_NUMERIK]
    invalid = df[required].isna().any(axis=1)
    if invalid.any():
        rows = (df.index[invalid] + 2).tolist()
        raise ValueError(
            f"Ditemukan nilai kosong/tidak valid pada {int(invalid.sum())} baris "
            f"(nomor baris sumber, perkiraan: {rows[:10]}). Periksa file sumber."
        )

    unknown = sorted(set(df["Jenis_Grafik"]) - set(URUTAN_GRAFIK))
    if unknown:
        raise ValueError(f"Kategori preset tidak dikenal: {unknown}")

    if not np.all(np.isclose(df["No"], np.round(df["No"]))):
        peringatan.append("Kolom No berisi angka pecahan; pastikan kolom ini merupakan nomor urut.")
    else:
        df["No"] = df["No"].astype(int)

    df["Jenis_Grafik"] = pd.Categorical(
        df["Jenis_Grafik"], categories=URUTAN_GRAFIK, ordered=True
    )
    df = df.sort_values("No").reset_index(drop=True)
    df["Run_Preset"] = df.groupby("Jenis_Grafik", observed=True).cumcount() + 1

    if not ((df["Min"] <= df["Avg"]) & (df["Avg"] <= df["Max"])).all():
        peringatan.append("Ada baris yang tidak memenuhi Min ≤ Average ≤ Max; periksa hasil benchmark mentah.")
    if (df["Low1"] > df["Avg"]).any():
        peringatan.append("Ada baris dengan 1% Low FPS lebih besar daripada Average FPS; periksa data sumber.")
    if df["No"].duplicated().any():
        peringatan.append("Nomor pengujian duplikat ditemukan.")
    if df.duplicated(subset=required).any():
        peringatan.append("Ada baris benchmark yang identik; verifikasi ke data mentah, jangan hapus otomatis.")

    counts = df["Jenis_Grafik"].value_counts().reindex(URUTAN_GRAFIK, fill_value=0)
    empty_groups = counts[counts == 0].index.tolist()
    if empty_groups:
        peringatan.append(f"Preset tanpa observasi: {', '.join(empty_groups)}.")

    return df, peringatan

# ============================================================
# 4. STATISTIKA DESKRIPTIF
# ============================================================
def hitung_modus(series: pd.Series) -> str:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return "Tidak ada data"
    frekuensi = s.value_counts()
    frekuensi_maks = int(frekuensi.max())
    if frekuensi_maks <= 1:
        return "Tidak ada modus (semua nilai unik)"
    modus = sorted(frekuensi[frekuensi == frekuensi_maks].index.tolist())
    return ", ".join(f"{nilai:g}" for nilai in modus)


def hitung_statistik_deskriptif(series: pd.Series) -> dict[str, Any]:
    """Menghitung ukuran pemusatan, penyebaran, bentuk distribusi, dan CV."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    n = int(s.count())
    if n == 0:
        return {ukuran: np.nan for ukuran in METRIK_STATISTIK}

    mean = float(s.mean())
    sd = float(s.std(ddof=1)) if n > 1 else np.nan
    var = float(s.var(ddof=1)) if n > 1 else np.nan
    q1 = float(s.quantile(0.25))
    q2 = float(s.quantile(0.50))
    q3 = float(s.quantile(0.75))
    minimum = float(s.min())
    maksimum = float(s.max())

    return {
        "Jumlah (Sum)": float(s.sum()),
        "Count (N)": n,
        "Mean": mean,
        "Modus": hitung_modus(s),
        "Median": q2,
        "Standard Error": sd / np.sqrt(n) if n > 1 else np.nan,
        "Standar Deviasi": sd,
        "Varians": var,
        "Q1": q1,
        "Q2": q2,
        "Q3": q3,
        "IQR": q3 - q1,
        "Skewness": float(s.skew()) if n >= 3 else np.nan,
        "Kurtosis (Excess)": float(s.kurt()) if n >= 4 else np.nan,
        "Minimum": minimum,
        "Maksimum": maksimum,
        "Range": maksimum - minimum,
        "CV (%)": (sd / mean * 100) if n > 1 and mean != 0 else np.nan,
    }


def fmt_num(value: Any, decimals: int = 2) -> str:
    if value is None or (isinstance(value, (float, np.floating)) and np.isnan(value)):
        return "—"
    if isinstance(value, (int, np.integer)):
        raw = f"{value:,}"
    elif isinstance(value, (float, np.floating)):
        raw = f"{value:,.{decimals}f}"
    else:
        return str(value)
    # Format angka Indonesia: pemisah ribuan titik, desimal koma.
    return raw.replace(",", "#").replace(".", ",").replace("#", ".")

# ============================================================
# 5. PEMBANGUN TABEL HTML (DENGAN SORTING)
# ============================================================
def buat_tabel_html(
    data: pd.DataFrame,
    table_id: str,
    class_name: str = "tabel",
    sortable: bool = True,
    preset_color: bool = False,
    number_decimals: int = 2,
    max_height: int | None = None,
) -> str:
    table_class = html.escape(class_name, quote=True)
    out = [f'<div class="data-panel"' + (f' style="max-height:{max_height}px;overflow:auto"' if max_height else '') + '>']
    out.append(f'<table class="{table_class}" id="{html.escape(table_id, quote=True)}"><thead><tr>')
    for idx, col in enumerate(data.columns):
        sort_attr = (
            f' class="sortable" onclick="cs2SortTable(\'{table_id}\',{idx})" title="Klik untuk mengurutkan"'
            if sortable else ""
        )
        out.append(f"<th{sort_attr}>{html.escape(str(col))}{' ⇳' if sortable else ''}</th>")
    out.append("</tr></thead><tbody>")

    for _, row in data.iterrows():
        out.append("<tr>")
        for col, value in row.items():
            sort_value: str
            if isinstance(value, (float, np.floating)):
                display_text = fmt_num(value, number_decimals)
                sort_value = str(float(value))
            elif isinstance(value, (int, np.integer)):
                display_text = fmt_num(value)
                sort_value = str(int(value))
            else:
                display_text = str(value)
                sort_value = display_text

            style = ""
            if preset_color and col in ("Jenis_Grafik", "Jenis Grafik"):
                preset = str(value)
                color = WARNA_GRAFIK.get(preset, "#222222")
                text_color = TEXT_GRAFIK.get(preset, "#FFFFFF")
                style = f' style="background:{color};color:{text_color};font-weight:800;text-align:center"'
            elif class_name == "tabel-deskriptif" and col != data.columns[0]:
                style = ' class="nilai"'

            out.append(
                f'<td data-sort="{html.escape(sort_value, quote=True)}"{style}>'
                f'{html.escape(display_text)}</td>'
            )
        out.append("</tr>")
    out.append("</tbody></table></div>")
    return "".join(out)

# ============================================================
# 6. VISUALISASI PLOTLY DENGAN TEMA ASLI GAMING AMBER
# ============================================================
WARNA_BG = "#151518"
WARNA_GRID = "#2B2B30"
WARNA_TEKS = "#EDEDEE"
WARNA_MUTED = "#A7A7AD"


def style_figure(fig: go.Figure, title: str, height: int = 440) -> go.Figure:
    fig.update_layout(
        title=dict(text=f"<b>{html.escape(title)}</b>", x=0.02, xanchor="left", font=dict(size=17, color="#FACC15")),
        template="plotly_dark",
        paper_bgcolor=WARNA_BG,
        plot_bgcolor=WARNA_BG,
        font=dict(family="Segoe UI, Arial, sans-serif", color=WARNA_TEKS, size=12),
        height=height,
        margin=dict(l=55, r=30, t=72, b=55),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hoverlabel=dict(bgcolor="#111114", font_color=WARNA_TEKS),
    )
    fig.update_xaxes(gridcolor=WARNA_GRID, zerolinecolor=WARNA_GRID, linecolor="#3A3A40")
    fig.update_yaxes(gridcolor=WARNA_GRID, zerolinecolor=WARNA_GRID, linecolor="#3A3A40", title_text="FPS")
    return fig


def buat_grafik(df: pd.DataFrame, metrik: str, tipe: str) -> go.Figure:
    """Membuat satu grafik untuk satu metrik; pilihan grafik dikendalikan UI."""
    judul_metrik = LABEL_METRIK[metrik]
    plot_df = df.copy()
    plot_df["Jenis_Grafik"] = plot_df["Jenis_Grafik"].astype(str)

    if tipe == "line":
        fig = px.line(
            plot_df, x="No", y=metrik, color="Jenis_Grafik", markers=True,
            category_orders={"Jenis_Grafik": URUTAN_GRAFIK},
            color_discrete_map=WARNA_GRAFIK,
            title=f"Grafik Garis — {judul_metrik} pada Seluruh Pengujian",
            hover_data={"No": True, "Jenis_Grafik": True, metrik: ":.2f"},
        )
        fig.update_traces(line=dict(width=2.4), marker=dict(size=7))
        fig.update_xaxes(title_text="Nomor Pengujian")
        return style_figure(fig, f"Grafik Garis: {judul_metrik}")

    if tipe == "bar":
        ringkasan = (
            plot_df.groupby("Jenis_Grafik", observed=False)[metrik]
            .agg(mean="mean", sd="std")
            .reindex(URUTAN_GRAFIK)
        )
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=URUTAN_GRAFIK,
            y=ringkasan["mean"],
            marker=dict(
                color=[WARNA_GRAFIK[preset] for preset in URUTAN_GRAFIK],
                line=dict(color="#0B0B0D", width=1),
            ),
            error_y=dict(
                type="data", array=ringkasan["sd"].fillna(0).tolist(),
                visible=True, color=WARNA_MUTED, thickness=1.5, width=5,
            ),
            text=[fmt_num(value, 2) for value in ringkasan["mean"]],
            textposition="outside", cliponaxis=False,
            hovertemplate="Preset: %{x}<br>Mean: %{y:.2f} FPS<br>Garis kesalahan: ±1 SD<extra></extra>",
        ))
        fig.update_xaxes(title_text="Preset Grafik", categoryorder="array", categoryarray=URUTAN_GRAFIK)
        fig.update_yaxes(title_text="Mean FPS ± 1 SD", automargin=True)
        fig.update_layout(showlegend=False, bargap=0.28)
        return style_figure(fig, f"Grafik Batang: Rata-rata {judul_metrik} ± Standar Deviasi")

    if tipe == "box":
        fig = px.box(
            plot_df, x="Jenis_Grafik", y=metrik, color="Jenis_Grafik", points="all",
            category_orders={"Jenis_Grafik": URUTAN_GRAFIK},
            color_discrete_map=WARNA_GRAFIK,
            title=f"Boxplot — Distribusi {judul_metrik}",
            hover_data={"No": True, "Jenis_Grafik": True, metrik: ":.2f"},
        )
        fig.update_traces(jitter=0.22, pointpos=0, marker=dict(size=5, opacity=0.78))
        fig.update_xaxes(title_text="Preset Grafik")
        fig.update_layout(showlegend=False)
        return style_figure(fig, f"Boxplot: Distribusi {judul_metrik}")

    raise ValueError(f"Jenis grafik tidak dikenal: {tipe}")

# ============================================================
# 7. MEMBANGUN DASHBOARD (DESAIN ASLI, KONTEN LEBIH LENGKAP)
# ============================================================
CSS_DASHBOARD = r"""
<style>
#dashboardCS2 {
    --bg-main:#0b0b0d; --bg-card:#151518; --bg-card2:#111114;
    --border:#2b2b30; --text:#ededee; --muted:#a7a7ad; --accent:#facc15;
    font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
    color:var(--text); background:var(--bg-main); padding:4px 0 18px;
}
#dashboardCS2 * { box-sizing:border-box; }
#dashboardCS2 .judul {
    background:linear-gradient(135deg,#1b1b1f 0%,#0d0d0f 100%);
    color:var(--text)!important; padding:24px 28px;
    border:1px solid var(--border); border-top:4px solid var(--accent);
    border-radius:12px; margin:4px 0 25px; box-shadow:0 8px 24px rgba(0,0,0,.22);
}
#dashboardCS2 .judul h1 {margin:0;font-size:23px;line-height:1.3;font-weight:800;color:var(--accent)!important;letter-spacing:.4px;}
#dashboardCS2 .judul p {margin:8px 0 0;font-size:13.5px;line-height:1.6;color:var(--muted)!important;}
#dashboardCS2 .subjudul {font-size:11px;letter-spacing:1.5px;font-weight:800;color:#f59e0b;text-transform:uppercase;margin-bottom:7px;}
#dashboardCS2 .bagian {
    background:#18181c;color:var(--accent)!important;padding:11px 16px;
    border:1px solid var(--border);border-left:4px solid var(--accent);border-radius:8px;
    margin:27px 0 12px!important;font-size:14px;font-weight:800;letter-spacing:.25px;
}
#dashboardCS2 .info {
    background:var(--bg-card2)!important;color:#dddddf!important;
    border:1px solid var(--border)!important;border-left:4px solid #eab308!important;
    padding:12px 16px;border-radius:8px;margin:10px 0 14px!important;
    font-size:13px;line-height:1.65;
}
#dashboardCS2 .info b, #dashboardCS2 .info strong {color:var(--accent)!important;}
#dashboardCS2 .ringkasan-grid {display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:10px;margin:13px 0 17px;}
#dashboardCS2 .ringkasan-card {background:linear-gradient(150deg,#17171b,#111114);border:1px solid var(--border);border-top:3px solid var(--accent);border-radius:9px;padding:14px 15px;min-width:0;}
#dashboardCS2 .ringkasan-label {font-size:10.5px;line-height:1.5;color:var(--muted);font-weight:750;text-transform:uppercase;letter-spacing:.45px;}
#dashboardCS2 .ringkasan-value {font-size:23px;line-height:1.3;color:var(--accent);font-weight:850;margin:5px 0 2px;overflow-wrap:anywhere;font-variant-numeric:tabular-nums;}
#dashboardCS2 .ringkasan-sub {font-size:10.5px;line-height:1.45;color:#a7a7ad;}
#dashboardCS2 .dua-kolom {display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:13px;margin:12px 0;}
#dashboardCS2 .panel {min-width:0;background:var(--bg-card);border:1px solid var(--border);border-radius:10px;padding:15px;}
#dashboardCS2 .panel h3 {font-size:13.5px;color:var(--accent);margin:0 0 10px;font-weight:800;}
#dashboardCS2 .panel p {font-size:12px;line-height:1.6;color:#c9c9ce;margin:0 0 10px;}
#dashboardCS2 .data-panel {overflow:auto;max-width:100%;border:1px solid var(--border);border-radius:9px;margin:10px 0;background:var(--bg-card);}
#dashboardCS2 table.tabel,#dashboardCS2 table.tabel-deskriptif {width:100%!important;border-collapse:separate!important;border-spacing:0!important;background:var(--bg-card)!important;color:var(--text)!important;margin:0!important;font-size:12px;font-variant-numeric:tabular-nums;}
#dashboardCS2 table.tabel {min-width:880px;}
#dashboardCS2 table.tabel-deskriptif {min-width:780px;font-size:12px;border:0!important;}
#dashboardCS2 table.tabel th,#dashboardCS2 table.tabel-deskriptif th {position:sticky;top:0;z-index:2;background:#1b1b20!important;color:var(--accent)!important;padding:10px 11px!important;text-align:center!important;font-weight:800!important;border:1px solid var(--border)!important;white-space:nowrap;}
#dashboardCS2 table.tabel th.sortable,#dashboardCS2 table.tabel-deskriptif th.sortable {cursor:pointer;user-select:none;}
#dashboardCS2 table.tabel th.sortable:hover,#dashboardCS2 table.tabel-deskriptif th.sortable:hover {background:#28282e!important;color:#fff!important;}
#dashboardCS2 table.tabel td,#dashboardCS2 table.tabel-deskriptif td {background:#131317;color:#dddddf;padding:8px 10px;text-align:center;border-bottom:1px solid #26262b;border-right:1px solid #242429;white-space:nowrap;}
#dashboardCS2 table.tabel tbody tr:nth-child(even) td,#dashboardCS2 table.tabel-deskriptif tbody tr:nth-child(even) td {background:#16161a;}
#dashboardCS2 table.tabel tbody tr:hover td,#dashboardCS2 table.tabel-deskriptif tbody tr:hover td {background:#202025;}
#dashboardCS2 table.tabel-deskriptif td.nilai {color:var(--accent)!important;font-weight:750!important;}
#dashboardCS2 .box-filter {display:flex;align-items:center;flex-wrap:wrap;gap:10px;margin:10px 0 12px;}
#dashboardCS2 .label-filter {color:var(--accent);font-weight:800;font-size:12px;}
#dashboardCS2 .select-filter,#dashboardCS2 .input-filter {padding:9px 12px;border:1px solid #404047;border-radius:7px;background:#111114;color:#fff;font-size:12px;font-weight:600;outline:none;min-width:180px;}
#dashboardCS2 .select-filter:focus,#dashboardCS2 .input-filter:focus {border-color:var(--accent);box-shadow:0 0 0 2px rgba(250,204,21,.12);}
#dashboardCS2 .radio-group-container {padding:14px 16px;margin:12px 0;background:var(--bg-card);border:1px solid var(--border);border-radius:10px;}
#dashboardCS2 .radio-group-title {display:block;margin:0 0 10px;color:var(--accent);font-size:13px;font-weight:800;}
#dashboardCS2 .radio-box-group {display:flex;flex-wrap:wrap;gap:9px;}
#dashboardCS2 .radio-box-item {display:flex;align-items:center;gap:8px;padding:9px 12px;border:1px solid #3b3b42;border-radius:7px;background:#111114;color:#d6d6da;font-size:12px;font-weight:700;cursor:pointer;user-select:none;transition:border-color .15s ease,background .15s ease,color .15s ease;}
#dashboardCS2 .radio-box-item:hover {border-color:var(--accent);color:#fff;}
#dashboardCS2 .radio-box-item input {width:15px;height:15px;accent-color:var(--accent);cursor:pointer;}
#dashboardCS2 .radio-box-item.active {border-color:var(--accent)!important;background:rgba(250,204,21,.09)!important;color:var(--accent)!important;}
#dashboardCS2 .chart-container {padding:7px;margin-top:10px;overflow:hidden;background:var(--bg-card);border:1px solid var(--border);border-radius:10px;}
#dashboardCS2 .chart-container .plotly-graph-div {width:100%!important;}
#dashboardCS2 .catatan {background:#111114;border:1px solid var(--border);border-left:4px solid #facc15;padding:13px 15px;border-radius:8px;margin:10px 0;color:#dedee2;font-size:12px;line-height:1.7;}
#dashboardCS2 .catatan strong {color:#facc15;}
#dashboardCS2 .quality-ok {color:#86efac;font-weight:800;}
#dashboardCS2 .quality-warn {color:#fde68a;font-weight:800;}
#dashboardCS2 .footer-note {color:#77777f;font-size:10.5px;margin-top:24px;padding-top:13px;border-top:1px solid var(--border);display:flex;flex-wrap:wrap;gap:8px;justify-content:space-between;}
#dashboardCS2 .small-note {font-size:11px;line-height:1.65;color:#92929a;margin:7px 0 0;}
@media(max-width:1050px){#dashboardCS2 .ringkasan-grid{grid-template-columns:repeat(3,minmax(0,1fr));}#dashboardCS2 .dua-kolom{grid-template-columns:1fr;}}
@media(max-width:650px){#dashboardCS2 .judul{padding:18px;}#dashboardCS2 .judul h1{font-size:19px;}#dashboardCS2 .ringkasan-grid{grid-template-columns:repeat(2,minmax(0,1fr));}#dashboardCS2 .info{padding:10px;}#dashboardCS2 .select-filter,#dashboardCS2 .input-filter{min-width:145px;width:100%;}}
</style>
"""


def _display_value(value: Any, col: str, decimals: int = 2) -> tuple[str, str]:
    if isinstance(value, (float, np.floating)):
        if np.isnan(value):
            return "—", ""
        if col in {"No", "Minimum FPS", "Maximum FPS", "Nilai Terisi", "Jumlah Pengujian"} and float(value).is_integer():
            return fmt_num(int(value)), str(float(value))
        return fmt_num(value, decimals), str(float(value))
    if isinstance(value, (int, np.integer)):
        return fmt_num(value), str(int(value))
    value_text = str(value)
    return value_text, value_text


def buat_tabel_html_berlabel(
    data: pd.DataFrame,
    table_id: str,
    class_name: str = "tabel",
    sortable: bool = True,
    preset_color: bool = False,
    number_decimals: int = 2,
) -> str:
    # Pembangun tabel mempertahankan fitur sorting seperti dashboard semula.
    out = [f'<div class="data-panel"><table class="{class_name}" id="{table_id}"><thead><tr>']
    for idx, col in enumerate(data.columns):
        sort_attr = f' class="sortable" onclick="cs2SortTable(\'{table_id}\',{idx})" title="Klik untuk mengurutkan"' if sortable else ""
        out.append(f'<th{sort_attr}>{html.escape(str(col))}{" ⇳" if sortable else ""}</th>')
    out.append("</tr></thead><tbody>")
    for _, row in data.iterrows():
        out.append("<tr>")
        for col, value in row.items():
            text, sort_value = _display_value(value, str(col), number_decimals)
            style = ""
            if preset_color and col in ("Jenis_Grafik", "Jenis Grafik", "Preset"):
                preset = str(value)
                bg = WARNA_GRAFIK.get(preset, "#222222")
                fg = TEXT_GRAFIK.get(preset, "#FFFFFF")
                style = f' style="background:{bg}!important;color:{fg}!important;font-weight:800;text-align:center"'
            elif class_name == "tabel-deskriptif" and col != data.columns[0]:
                style = ' class="nilai"'
            out.append(f'<td data-sort="{html.escape(sort_value, quote=True)}"{style}>{html.escape(text)}</td>')
        out.append("</tr>")
    out.append("</tbody></table></div>")
    return "".join(out)


def _buat_tabel_ringkasan_means(df: pd.DataFrame) -> pd.DataFrame:
    means = df.groupby("Jenis_Grafik", observed=False)[KOLOM_NUMERIK].mean().reindex(URUTAN_GRAFIK)
    means.index.name = "Jenis Grafik"
    means = means.rename(columns=LABEL_METRIK).reset_index()
    return means


def _buat_tabel_frekuensi(df: pd.DataFrame) -> pd.DataFrame:
    freq = df["Jenis_Grafik"].value_counts().reindex(URUTAN_GRAFIK, fill_value=0)
    total = int(len(df))
    return pd.DataFrame({
        "Jenis Grafik": URUTAN_GRAFIK,
        "Jumlah Pengujian": [int(freq[p]) for p in URUTAN_GRAFIK],
        "Persentase (%)": [float(freq[p] / total * 100) if total else 0.0 for p in URUTAN_GRAFIK],
    })


def _buat_tabel_tipe_data(df: pd.DataFrame, sumber_kolom: dict[str, str]) -> pd.DataFrame:
    data = [
        [sumber_kolom.get("No", "No"), "Identitas/nomor urut", "Tidak dianalisis sebagai variabel pengukuran", int(df["No"].count())],
        [sumber_kolom.get("Jenis_Grafik", "Jenis_Grafik"), "Ordinal", "Kategori kualitas grafik berurutan: Low → Medium → High → Very High", int(df["Jenis_Grafik"].count())],
        [sumber_kolom.get("Min", "MinimumFramePerSeconds"), "Rasio (numerik)", "Nilai minimum FPS pada satu pengujian", int(df["Min"].count())],
        [sumber_kolom.get("Max", "MaximumFramePerSeconds"), "Rasio (numerik)", "Nilai maksimum FPS pada satu pengujian", int(df["Max"].count())],
        [sumber_kolom.get("Avg", "AverageFramePerSeconds"), "Rasio (numerik)", "Nilai Average FPS pada satu pengujian", int(df["Avg"].count())],
        [sumber_kolom.get("Low1", "1%FramesPerSecond"), "Rasio (numerik)", "Nilai 1% Low FPS pada satu pengujian", int(df["Low1"].count())],
    ]
    return pd.DataFrame(data, columns=["Kolom Sumber", "Tipe Data", "Deskripsi Variabel", "Nilai Terisi"])


def _buat_tabel_quality(df: pd.DataFrame, warnings: list[str]) -> pd.DataFrame:
    missing = int(df[["No", "Jenis_Grafik", *KOLOM_NUMERIK]].isna().sum().sum())
    count_by = df["Jenis_Grafik"].value_counts().reindex(URUTAN_GRAFIK, fill_value=0)
    checks = [
        ["Jumlah observasi", f"{len(df)} baris", "Jumlah data yang dibaca"],
        ["Nilai kosong/tidak valid", str(missing), "0 berarti seluruh kolom yang dipakai analisis terisi"],
        ["Nomor pengujian duplikat", str(int(df["No"].duplicated().sum())), "Nomor digunakan sebagai identitas baris"],
        ["Baris benchmark identik", str(int(df[["No", "Jenis_Grafik", *KOLOM_NUMERIK]].duplicated().sum())), "Tidak dihapus otomatis"],
        ["Konsistensi Min ≤ Avg ≤ Max", "Sesuai" if bool(((df["Min"] <= df["Avg"]) & (df["Avg"] <= df["Max"])).all()) else "Perlu diperiksa", "Pemeriksaan nilai antar kolom FPS"],
        ["Konsistensi 1% Low ≤ Average", "Sesuai" if bool((df["Low1"] <= df["Avg"]).all()) else "Perlu diperiksa", "Pemeriksaan hubungan nilai FPS"],
        ["Jumlah tiap preset", " · ".join(f"{p}: {int(count_by[p])}" for p in URUTAN_GRAFIK), "Distribusi observasi per kategori"],
        ["Peringatan validasi", str(len(warnings)), "Lihat pesan di bagian bawah jika ada peringatan"],
    ]
    return pd.DataFrame(checks, columns=["Pemeriksaan", "Hasil", "Keterangan"])


def render_dashboard(df: pd.DataFrame, filename: str, sheet_name: str | None,
                     warnings: list[str], source_columns: dict[str, str] | None = None,
                     standalone: bool = False) -> str:
    source_columns = source_columns or {}
    frekuensi = df["Jenis_Grafik"].value_counts().reindex(URUTAN_GRAFIK, fill_value=0)
    summary_means = _buat_tabel_ringkasan_means(df)
    freq_table = _buat_tabel_frekuensi(df)
    type_table = _buat_tabel_tipe_data(df, source_columns)
    quality_table = _buat_tabel_quality(df, warnings)

    # Mean dan pola penyebaran dihitung pada tiap preset, tanpa menguji signifikansi.
    means_by_preset = df.groupby("Jenis_Grafik", observed=False)[KOLOM_NUMERIK].mean().reindex(URUTAN_GRAFIK)
    avg_means = means_by_preset["Avg"]
    low_means = means_by_preset["Low1"]
    best_avg = str(avg_means.idxmax())
    lowest_avg = str(avg_means.idxmin())
    best_low = str(low_means.idxmax())
    lowest_low = str(low_means.idxmin())

    total = int(len(df))
    n_presets = int(df["Jenis_Grafik"].nunique())
    counts = [int(frekuensi[p]) for p in URUTAN_GRAFIK]
    equal_counts = len(set(counts)) == 1
    runs_text = f"{counts[0]} pengujian per preset" if equal_counts and counts else "Jumlah pengujian per preset tidak sama"
    shape_text = f"{total} baris × 6 kolom"
    missing_count = int(df[["No", "Jenis_Grafik", *KOLOM_NUMERIK]].isna().sum().sum())

    # Seluruh statistik: setiap metrik mempunyai satu panel dengan keseluruhan data + empat preset.
    stats_panels: list[str] = []
    for metric in KOLOM_NUMERIK:
        stat_keys = METRIK_STATISTIK
        columns = ["Keseluruhan", *URUTAN_GRAFIK]
        stats = {"Keseluruhan": hitung_statistik_deskriptif(df[metric])}
        for preset in URUTAN_GRAFIK:
            stats[preset] = hitung_statistik_deskriptif(df.loc[df["Jenis_Grafik"] == preset, metric])
        rows = []
        for key in stat_keys:
            rows.append({"Ukuran Statistik": key, **{col: stats[col][key] for col in columns}})
        table = pd.DataFrame(rows)
        stats_panels.append(
            f'<div class="stats-panel" id="stats-{metric}" style="display:{"block" if metric == "Avg" else "none"}">'
            + buat_tabel_html_berlabel(table, f"tabelStatistik{metric}", class_name="tabel-deskriptif", sortable=True)
            + "</div>"
        )

    # Tabel sumber dengan nama yang mudah dibaca, namun urutan dan nilai tetap sumber asli.
    data_tampil = df[["No", "Jenis_Grafik", *KOLOM_NUMERIK]].copy()
    data_tampil["Jenis_Grafik"] = data_tampil["Jenis_Grafik"].astype(str)
    data_tampil.columns = ["No", "Jenis Grafik", "Minimum FPS", "Maximum FPS", "Average FPS", "1% Low FPS"]
    raw_table = buat_tabel_html_berlabel(data_tampil, "tabelBenchmarkCS2", class_name="tabel", preset_color=True, sortable=True)
    freq_html = buat_tabel_html_berlabel(freq_table, "tabelFrekuensi", class_name="tabel-deskriptif", preset_color=True)
    mean_html = buat_tabel_html_berlabel(summary_means, "tabelMeanPreset", class_name="tabel-deskriptif", preset_color=True)
    type_html = buat_tabel_html_berlabel(type_table, "tabelTipeData", class_name="tabel-deskriptif", sortable=False)
    quality_html = buat_tabel_html_berlabel(quality_table, "tabelValidasi", class_name="tabel-deskriptif", sortable=False)

    # Kartu ringkasan menjaga gaya dashboard asli, tanpa menggantikan detail tabel.
    kpi_html = f"""
    <div class="ringkasan-grid">
      <div class="ringkasan-card" style="border-top-color:#FACC15"><div class="ringkasan-label">Total Observasi</div><div class="ringkasan-value">{total}</div><div class="ringkasan-sub">{shape_text}</div></div>
      <div class="ringkasan-card" style="border-top-color:#F59E0B"><div class="ringkasan-label">Preset Grafik</div><div class="ringkasan-value">{n_presets}</div><div class="ringkasan-sub">Low sampai Very High</div></div>
      <div class="ringkasan-card" style="border-top-color:#EA580C"><div class="ringkasan-label">Pengulangan per Preset</div><div class="ringkasan-value">{counts[0] if equal_counts and counts else '—'}</div><div class="ringkasan-sub">{html.escape(runs_text)}</div></div>
      <div class="ringkasan-card" style="border-top-color:#FACC15"><div class="ringkasan-label">Mean Average FPS Tertinggi</div><div class="ringkasan-value">{fmt_num(float(avg_means.max()), 2)}</div><div class="ringkasan-sub">Preset {html.escape(best_avg)}</div></div>
      <div class="ringkasan-card" style="border-top-color:#EF4444"><div class="ringkasan-label">Mean 1% Low Tertinggi</div><div class="ringkasan-value">{fmt_num(float(low_means.max()), 2)}</div><div class="ringkasan-sub">Preset {html.escape(best_low)}</div></div>
    </div>
    """

    # Temuan otomatis bersifat deskriptif, berbasis angka pada dataset ini saja.
    avg_sd = {p: hitung_statistik_deskriptif(df.loc[df["Jenis_Grafik"] == p, "Avg"]) for p in URUTAN_GRAFIK}
    low_sd = {p: hitung_statistik_deskriptif(df.loc[df["Jenis_Grafik"] == p, "Low1"]) for p in URUTAN_GRAFIK}
    cv_avg = {p: avg_sd[p]["CV (%)"] for p in URUTAN_GRAFIK}
    cv_low = {p: low_sd[p]["CV (%)"] for p in URUTAN_GRAFIK}
    preset_cv_avg = min(cv_avg, key=lambda p: cv_avg[p])
    preset_cv_low = min(cv_low, key=lambda p: cv_low[p])
    lines = [
        f"<li><strong>Average FPS:</strong> mean tertinggi ditemukan pada preset {html.escape(best_avg)} ({fmt_num(float(avg_means.max()), 2)} FPS), sedangkan mean terendah pada {html.escape(lowest_avg)} ({fmt_num(float(avg_means.min()), 2)} FPS).</li>",
        f"<li><strong>1% Low FPS:</strong> mean tertinggi ditemukan pada preset {html.escape(best_low)} ({fmt_num(float(low_means.max()), 2)} FPS), sedangkan mean terendah pada {html.escape(lowest_low)} ({fmt_num(float(low_means.min()), 2)} FPS).</li>",
        f"<li><strong>Penyebaran Average FPS:</strong> berdasarkan CV deskriptif, preset {html.escape(preset_cv_avg)} mempunyai variasi relatif terendah ({fmt_num(float(cv_avg[preset_cv_avg]), 2)}%). Ini adalah gambaran penyebaran pada data yang tersedia, bukan uji signifikansi.</li>",
        f"<li><strong>Penyebaran 1% Low FPS:</strong> berdasarkan CV deskriptif, preset {html.escape(preset_cv_low)} mempunyai variasi relatif terendah ({fmt_num(float(cv_low[preset_cv_low]), 2)}%). Baca bersama standar deviasi dan IQR sebelum menafsirkan konsistensi.</li>",
    ]
    interpretation = "<ul>" + "".join(lines) + "</ul>"

    warning_block = ""
    if warnings:
        warning_block = "<div class='catatan'><strong>Perlu diperiksa:</strong><ul>" + "".join(f"<li>{html.escape(w)}</li>" for w in warnings) + "</ul></div>"
    else:
        warning_block = "<div class='catatan'><strong>Status validasi:</strong> Pemeriksaan dasar tidak menemukan nilai kosong, kategori tidak dikenal, atau pelanggaran kondisi Min ≤ Avg ≤ Max dan 1% Low ≤ Average pada dataset yang dibaca.</div>"

    # Bangun 12 grafik (4 metrik × 3 tipe) untuk dipilih dari kontrol asli dropdown + radio box.
    chart_html: list[str] = []
    first_chart = True
    for metric in ["Avg", "Low1", "Min", "Max"]:
        for chart_type in ["line", "bar", "box"]:
            fig = buat_grafik(df, metric, chart_type)
            include_plotly = "cdn" if first_chart else False
            fig_html = fig.to_html(
                full_html=False,
                include_plotlyjs=include_plotly,
                config={"responsive": True, "displaylogo": False, "scrollZoom": True},
                div_id=f"plotly-{metric}-{chart_type}",
            )
            first_chart = False
            visible = (metric == "Avg" and chart_type == "line")
            chart_html.append(
                f'<div class="chart-container" id="chart-{metric}-{chart_type}" style="display:{"block" if visible else "none"}">{fig_html}</div>'
            )

    chart_metric_options = "".join(
        f'<option value="{m}" {"selected" if m == "Avg" else ""}>{html.escape(LABEL_METRIK[m])}</option>'
        for m in ["Avg", "Low1", "Min", "Max"]
    )
    stats_metric_options = "".join(
        f'<option value="{m}" {"selected" if m == "Avg" else ""}>{html.escape(LABEL_METRIK[m])}</option>'
        for m in ["Avg", "Low1", "Min", "Max"]
    )

    dashboard_fragment = f"""
    {CSS_DASHBOARD}
    <div id="dashboardCS2">
      <header class="judul">
        <div class="subjudul">Data Lab · Counter-Strike 2</div>
        <h1>BENCHMARK PERFORMANCE CS2 — STATISTIKA DESKRIPTIF</h1>
        <p>Dashboard interaktif untuk membaca data benchmark berdasarkan preset Low, Medium, High, dan Very High. Tampilan mempertahankan gaya Gaming Amber dari dashboard sebelumnya, dengan tabel yang dapat difilter/diurutkan serta visualisasi yang dapat dipilih.</p>
        <p style="margin-top:10px"><strong>File:</strong> {html.escape(Path(filename).name)} &nbsp; · &nbsp; <strong>Sheet:</strong> {html.escape(str(sheet_name or '—'))} &nbsp; · &nbsp; <strong>Fokus:</strong> Statistik deskriptif</p>
      </header>

      <div class="bagian">1. RINGKASAN DATA BENCHMARK</div>
      {kpi_html}
      <div class="info">Dataset berisi <b>{total} observasi</b> dan <b>6 kolom</b>. Variabel pengukuran numerik yang diolah adalah Minimum FPS, Maximum FPS, Average FPS, dan 1% Low FPS; preset grafik menjadi kategori pengelompokan. Seluruh angka pada dashboard dihitung dari file yang dipilih.</div>
      <div class="catatan"><strong>Catatan validasi metrik:</strong> Pastikan kolom 1% Low FPS benar-benar berisi hasil pengukuran 1% Low dari sumber benchmark, bukan hanya kolom 0.1% Low FPS yang diganti namanya.</div>

      <div class="bagian">2. SUMBER DATA DAN TIPE VARIABEL</div>
      <div class="info"><b>Sumber data:</b> file benchmark CS2 yang dipilih pengguna. Untuk workbook Excel, sheet <b>{html.escape(str(sheet_name or '—'))}</b> digunakan. Tabel di bawah menuliskan deskripsi kolom, tipe data sesuai klasifikasi tugas, dan jumlah nilai terisi.</div>
      {type_html}
      <div class="dua-kolom">
        <div class="panel"><h3>Distribusi Pengujian</h3><p>Frekuensi menunjukkan banyaknya observasi pada setiap preset. Persentase dihitung terhadap seluruh baris yang valid.</p>{freq_html}</div>
        <div class="panel"><h3>Rata-rata Setiap Variabel</h3><p>Tabel berikut merangkum mean dari seluruh variabel numerik pada setiap preset. Gunakan sebagai ikhtisar, kemudian periksa tabel statistik lengkap di bagian berikutnya.</p>{mean_html}</div>
      </div>

      <div class="bagian">3. DATA HASIL BENCHMARK CS2</div>
      <div class="info"><b>Filter:</b> cari teks/angka atau pilih preset. <b>Sorting:</b> klik judul kolom untuk mengurutkan naik/turun. Filter hanya mengubah tampilan tabel dan tidak mengubah perhitungan statistik.</div>
      <div class="box-filter">
        <label class="label-filter" for="cs2Search">Cari data:</label>
        <input id="cs2Search" class="input-filter" type="search" placeholder="Contoh: Low atau 260.07" oninput="cs2FilterTable()">
        <label class="label-filter" for="cs2PresetFilter">Jenis grafik:</label>
        <select id="cs2PresetFilter" class="select-filter" onchange="cs2FilterTable()">
          <option value="ALL">Semua preset</option><option value="Low">Low</option><option value="Medium">Medium</option><option value="High">High</option><option value="Very High">Very High</option>
        </select>
        <span class="label-filter" id="cs2VisibleRows">{total} baris</span>
      </div>
      {raw_table}
      <div class="small-note">Nomor No diperlakukan sebagai identitas/urutan pengujian, bukan variabel FPS. Nilai sumber tidak dihapus atau diubah kecuali normalisasi tipe data dan nama kolom untuk keperluan analisis.</div>

      <div class="bagian">4. VISUALISASI DATA INTERAKTIF</div>
      <div class="info">Pilih metrik yang ingin diamati melalui dropdown, lalu pilih jenis visualisasi melalui radio box. Grafik garis menampilkan hasil tiap pengujian, grafik batang membandingkan mean dengan garis kesalahan ±1 standar deviasi, sedangkan boxplot menunjukkan median, kuartil, sebaran, dan calon pencilan.</div>
      <div class="radio-group-container">
        <div class="box-filter"><span class="radio-group-title" style="margin:0">Pilih variabel FPS:</span>
          <select id="cs2MetricChart" class="select-filter" onchange="cs2UpdateChart()">{chart_metric_options}</select>
        </div>
        <span class="radio-group-title">Pilih jenis grafik:</span>
        <div class="radio-box-group">
          <label class="radio-box-item active" id="cs2LabelLine"><input type="radio" name="cs2ChartType" value="line" checked onclick="cs2SetChartType('line')"> Grafik Garis (Line Chart)</label>
          <label class="radio-box-item" id="cs2LabelBar"><input type="radio" name="cs2ChartType" value="bar" onclick="cs2SetChartType('bar')"> Grafik Batang (Mean ± SD)</label>
          <label class="radio-box-item" id="cs2LabelBox"><input type="radio" name="cs2ChartType" value="box" onclick="cs2SetChartType('box')"> Boxplot (Distribusi)</label>
        </div>
      </div>
      {''.join(chart_html)}
      <div class="small-note">Catatan: garis ±1 SD pada grafik batang adalah penanda penyebaran deskriptif, bukan interval kepercayaan. Titik di luar whisker boxplot adalah calon pencilan dan tidak otomatis berarti data salah.</div>

      <div class="bagian">5. STATISTIKA DESKRIPTIF LENGKAP</div>
      <div class="info">Pilih metrik untuk melihat statistik <b>keseluruhan 60 observasi</b> dan statistik pada masing-masing preset. Tabel mencakup jumlah (sum), count, mean, modus, median, standard error, standar deviasi, varians, Q1–Q3, IQR, skewness, kurtosis, minimum, maksimum, range, dan CV.</div>
      <div class="box-filter"><label class="label-filter" for="cs2MetricStats">Pilih metrik statistik:</label><select id="cs2MetricStats" class="select-filter" onchange="cs2SwitchStats(this.value)">{stats_metric_options}</select></div>
      {''.join(stats_panels)}
      <div class="catatan"><strong>Catatan perhitungan:</strong> Median sama dengan Q2. Varians dan standar deviasi menggunakan rumus sampel (ddof = 1); standard error = SD/√n; IQR = Q3 − Q1; CV = SD/mean × 100%; kurtosis ditampilkan sebagai <i>excess kurtosis</i>. Bila semua nilai pada satu kelompok unik, modus ditampilkan sebagai tidak ada modus tunggal.</div>

      <div class="bagian">6. INTERPRETASI HASIL SECARA DESKRIPTIF</div>
      <div class="info">Ringkasan di bawah dihasilkan otomatis dari angka pada dataset yang dipilih. Pernyataan menggambarkan data pengujian ini saja dan tidak dimaksudkan sebagai kesimpulan universal untuk seluruh perangkat.</div>
      <div class="catatan">{interpretation}</div>

      <div class="bagian">7. PEMERIKSAAN KUALITAS DATA</div>
      <div class="info">Pemeriksaan berikut membantu memastikan data siap diringkas. Apabila terdapat peringatan, periksa hasil benchmark mentah terlebih dahulu; program tidak menghapus data secara otomatis.</div>
      {quality_html}
      {warning_block}
      <div class="footer-note"><span>CS2 Benchmark · Dashboard Statistika Deskriptif</span><span>Desain: Gaming Amber · kuning, amber, oranye, merah</span></div>
    </div>

    <script>
    (function() {{
      const root = document.getElementById('dashboardCS2');
      if (!root) return;

      window.cs2FilterTable = function() {{
        const search = (root.querySelector('#cs2Search')?.value || '').toLowerCase();
        const preset = root.querySelector('#cs2PresetFilter')?.value || 'ALL';
        const table = root.querySelector('#tabelBenchmarkCS2');
        if (!table) return;
        let visible = 0;
        table.querySelectorAll('tbody tr').forEach(row => {{
          const text = row.textContent.toLowerCase();
          const category = (row.cells[1]?.textContent || '').trim();
          const show = text.includes(search) && (preset === 'ALL' || category === preset);
          row.style.display = show ? '' : 'none';
          if (show) visible++;
        }});
        const badge = root.querySelector('#cs2VisibleRows');
        if (badge) badge.textContent = visible + ' baris terlihat';
      }};

      const sortDirections = {{}};
      window.cs2SortTable = function(tableId, colIndex) {{
        const table = root.querySelector('#' + tableId);
        if (!table) return;
        const tbody = table.querySelector('tbody');
        const rows = Array.from(tbody.querySelectorAll('tr'));
        const key = tableId + '-' + colIndex;
        sortDirections[key] = !sortDirections[key];
        const asc = sortDirections[key];
        rows.sort((a,b) => {{
          const av = a.cells[colIndex]?.dataset.sort ?? a.cells[colIndex]?.textContent ?? '';
          const bv = b.cells[colIndex]?.dataset.sort ?? b.cells[colIndex]?.textContent ?? '';
          const an = Number(av), bn = Number(bv);
          if (av.trim() !== '' && bv.trim() !== '' && Number.isFinite(an) && Number.isFinite(bn)) return asc ? an-bn : bn-an;
          return asc ? av.localeCompare(bv) : bv.localeCompare(av);
        }});
        rows.forEach(row => tbody.appendChild(row));
        if (tableId === 'tabelBenchmarkCS2') window.cs2FilterTable();
      }};

      let metricSelected = 'Avg';
      let chartSelected = 'line';
      window.cs2SetChartType = function(chartType) {{
        chartSelected = chartType;
        ['line','bar','box'].forEach(t => {{
          const label = root.querySelector('#cs2Label' + t.charAt(0).toUpperCase()+t.slice(1));
          if (label) label.classList.toggle('active', t === chartSelected);
        }});
        window.cs2UpdateChart();
      }};
      window.cs2UpdateChart = function() {{
        const selector = root.querySelector('#cs2MetricChart');
        if (selector) metricSelected = selector.value;
        root.querySelectorAll('.chart-container').forEach(panel => panel.style.display = 'none');
        const active = root.querySelector('#chart-' + metricSelected + '-' + chartSelected);
        if (active) {{
          active.style.display = 'block';
          setTimeout(() => {{
            if (window.Plotly) active.querySelectorAll('.js-plotly-plot').forEach(chart => Plotly.Plots.resize(chart));
          }}, 80);
        }}
      }};
      window.cs2SwitchStats = function(metric) {{
        root.querySelectorAll('.stats-panel').forEach(panel => panel.style.display = 'none');
        const panel = root.querySelector('#stats-' + metric);
        if (panel) panel.style.display = 'block';
      }};
    }})();
    </script>
    """

    if standalone:
        return "<!DOCTYPE html><html lang='id'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width, initial-scale=1'><title>CS2 Benchmark — Statistika Deskriptif</title></head><body style='margin:0;padding:18px;background:#0b0b0d'>" + dashboard_fragment + "</body></html>"
    return dashboard_fragment

# ============================================================
# 8. JALANKAN DASHBOARD
# ============================================================
def jalankan_dashboard() -> tuple[pd.DataFrame, str]:
    filename = pilih_file_dataset()
    raw_df, sheet_name, source_columns = baca_dataset(filename)
    df, warnings = validasi_dataset(raw_df)
    dashboard_html = render_dashboard(df, filename, sheet_name, warnings, source_columns, standalone=False)
    display(HTML(dashboard_html))
    return df, dashboard_html


def simpan_preview_html(input_path: str, output_path: str) -> str:
    raw_df, sheet_name, source_columns = baca_dataset(input_path)
    df, warnings = validasi_dataset(raw_df)
    page = render_dashboard(df, input_path, sheet_name, warnings, source_columns, standalone=True)
    Path(output_path).write_text(page, encoding="utf-8")
    return output_path


if __name__ == "__main__":
    # Saat dijalankan pada notebook/Colab, semua tabel dan grafik ditampilkan sebagai satu dashboard.
    jalankan_dashboard()


Saving 3TIA_06_DataBenchmarkCS2.xlsx to 3TIA_06_DataBenchmarkCS2.xlsx
